In [ ]:
!pip -q install transformers torch gradio

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# Pretrained sentiment analysis model
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

""""RoBERTa-base (a transformer language model)
Fine-tuned for sentiment analysis
Trained mainly on Twitter/X posts, so it understands informal language, emojis, hashtags, etc"""

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
"""
| Task                     | Model Class                          |
| ------------------------ | ------------------------------------ |
| Text classification      | `AutoModelForSequenceClassification` |
| Text generation          | `AutoModelForCausalLM`               |
| Question answering       | `AutoModelForQuestionAnswering`      |
| Named Entity Recognition | `AutoModelForTokenClassification`    |
"""

labels = ["Negative 😠", "Neutral 😐", "Positive 😊"]

"""the model outputs class indices:
0
1
2
Without this list, the prediction would simply be a number."""

print("Model loaded successfully!")

Loading model...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [ ]:
def predict_sentiment(text):

    if len(text.strip()) == 0:
        return {"Error": 1.0}

    """
    strip() removes spaces from the beginning and end.

    Example:
    "   Hello   ".strip() -> "Hello"

    "        ".Strip()  -> ""
    """
    # Convert text into tokens
    inputs = tokenizer(
        text,
        return_tensors="pt", #Returns PyTorch tensors instead of Python lists,Without it:input_ids = [0, 314, 56, 2],With it:tensor([[0, 314, 56, 2]]).Since the model is implemented in PyTorch, it expects tensors.
        truncation=True,
        padding=True
    )

    # Model inference
    with torch.no_grad(): #only making predictions, not training.
        outputs = model(**inputs)
        """
        A logit is simply a raw score that the neural network assigns to each possible class before converting them into probabilities.

        Think of them as votes.

        Example:
        Sentiment	Logit
        Negative	-1.3
        Neutral	0.8
        Positive	3.5

        The largest logit (3.5) indicates that the model currently favors the Positive class.
        """

    # Convert logits to probabilities
    probs = torch.softmax(outputs.logits, dim=1)[0]

    return {
        labels[0]: float(probs[0]),
        labels[1]: float(probs[1]),
        labels[2]: float(probs[2]),
    }

In [ ]:
predict_sentiment("happy")

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

{'Negative 😠': 0.03359246253967285,
 'Neutral 😐': 0.14152473211288452,
 'Positive 😊': 0.8248828053474426}

In [ ]:
demo = gr.Interface(
    fn=predict_sentiment,

    inputs=gr.Textbox(
        lines=5,
        placeholder="Type your sentence here..."
    ),

    outputs=gr.Label(num_top_classes=3),

    title="😊 Transformer Sentiment Analyzer",

    description="""
Enter any sentence and the Transformer model
will predict whether it is Positive, Neutral, or Negative.
""",

    examples=[
        ["I love this movie!"],
        ["This is the worst product I have ever bought."],
        ["The weather is okay today."],
        ["I am very happy today."],
        ["The food was terrible."]
    ]
)

In [ ]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65074dc9dadbd4cb5b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
